# High-Fidelity Document Reconstruction & Raw Stream Extraction

## 1. Objective
This notebook executes a raw binary stream extraction pipeline from dense scanned PDFs. It bypasses conventional document rasterization overhead to isolate original embedded image streams (chunks) and geometrically reconstruct them onto a lossless, pixel-perfect canvas optimized for high-accuracy OCR text extraction.

---

## 2. Technical Challenges & Elegant Solutions

### Challenge A: Destructive Black Backgrounds (Inverted Binary Masks)
Certain scanned PDF systems compress textual bodies as monochromatic binary image masks. When extracted raw, standard image decoders fail to interpret their native transparent channels, rendering the background solid black and the text white.
*   **Solution (Bitwise Polarity Inversion):** 
    The pipeline converts the decoded chunk to grayscale in memory and measures its mean pixel intensity $\mu$. If $\mu < 127$ (indicating a dominantly dark, inverted mask), it dynamically triggers a bitwise NOT operation:
    $$\text{Image}_{\text{clean}} = \neg \text{Image}_{\text{raw}}$$
    This cleanly flips the mask to solid black text on a pure white background before writing it to disk.

### Challenge B: Layer Erasing (Opaque White Overwriting)
Stitching multiple raw image layers directly using standard canvas assignments overwrites underlying content. Larger white-background canvas components (such as base metadata forms or logo margins) frequently wipe out previously drawn textual fragments, tables, and punctuation marks.
*   **Solution (Intelligent Darken/Multiply Layer Fusion):**
    Instead of utilizing destructive write operations, the pipeline enforces a set-theoretic minimum pixel blend using OpenCV and NumPy:
    $$\text{Canvas}(x, y) = \min\left(\text{Canvas}(x, y), \text{Incoming Chunk}(x, y)\right)$$
    Since pure white pixels are represented by $255$ and dark text pixels by values near $0$, this mathematical constraint ensures that the darkest value (the text ink) always overrides the white space, enabling flawless multi-layer blending without content erasure.

### Challenge C: Geometric Layout & Resolution Mismatches
Individual assets (logos vs. text grids) inside the PDF are stored at drastically different Native DPI levels. Forcing a direct 1:1 pixel copy causes scale drift, misalignment, and text blurring.
*   **Solution (Max-Scale Coordinate Tracking):**
    Before stitching, the pipeline runs an exploratory scan of all elements on the page to calculate the absolute maximum native scaling ratio on both axes:
    $$\text{Scale}_x = \frac{\text{Width}_{\text{pixels}}}{\text{Width}_{\text{points}}}, \quad \text{Scale}_y = \frac{\text{Height}_{\text{pixels}}}{\text{Height}_{\text{points}}}$$
    The master canvas is initialized at this maximum scale. Smaller assets are dynamically rescaled to their exact spatial coordinates using `cv2.INTER_LANCZOS4` interpolation, ensuring crisp edges and absolute layout integrity.

In [ ]:
import fitz  # PyMuPDF
import cv2
import numpy as np
import os

def try_pdfimages_perfect_blend_stitching(pdf_path, output_folder="pdfimages_raw_output"):
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)
        print(f"[+] Created main output folder: {output_folder}")

    doc = fitz.open(pdf_path)
    print(f"[+] Opened PDF. Total pages: {len(doc)}")
    print("[+] Mode: INTELLIGENT PIXEL BLEND (Darken/Multiply Layer Fusion)")
    print("="*85)

    total_img_count = 0
    
    for page_index in range(len(doc)):
        page = doc.load_page(page_index)
        image_list = page.get_images()
        
        if image_list:
            page_num = page_index + 1
            page_folder = os.path.join(output_folder, f"page_{page_num}")
            os.makedirs(page_folder, exist_ok=True)
            
            max_scale_x = 1.0
            max_scale_y = 1.0
            
            for img in image_list:
                xref = img[0]
                rects = page.get_image_rects(xref)
                if rects:
                    rect = rects[0]
                    scale_x = img[2] / rect.width
                    scale_y = img[3] / rect.height
                    if scale_x > max_scale_x: max_scale_x = scale_x
                    if scale_y > max_scale_y: max_scale_y = scale_y

            canvas_w = int(page.rect.width * max_scale_x)
            canvas_h = int(page.rect.height * max_scale_y)
            canvas = np.ones((canvas_h, canvas_w, 3), dtype=np.uint8) * 255
            
            page_img_count = 0
            for img in image_list:
                xref = img[0]
                
                base_image = doc.extract_image(xref)
                image_bytes = base_image["image"]
                image_ext = base_image["ext"]

                page_img_count += 1
                total_img_count += 1
                
                chunk_np = np.frombuffer(image_bytes, dtype=np.uint8)
                chunk_img = cv2.imdecode(chunk_np, cv2.IMREAD_COLOR)
                
                if chunk_img is None:
                    continue
                
                gray_chunk = cv2.cvtColor(chunk_img, cv2.COLOR_BGR2GRAY)
                if np.mean(gray_chunk) < 127:
                    chunk_img = cv2.bitwise_not(chunk_img)
                
                chunk_output_path = os.path.join(page_folder, f"clean_chunk_{page_img_count}.png")
                cv2.imwrite(chunk_output_path, chunk_img)
                print(f"    [✓] Page {page_num} -> Saved Crisp Chunk: clean_chunk_{page_img_count}.png")
                
                rects = page.get_image_rects(xref)
                for rect in rects:
                    x0 = int(rect.x0 * max_scale_x)
                    y0 = int(rect.y0 * max_scale_y)
                    
                    target_w = int(rect.x1 * max_scale_x) - x0
                    target_h = int(rect.y1 * max_scale_y) - y0
                    
                    if target_w <= 0 or target_h <= 0:
                        continue
                    
                    resized_chunk = cv2.resize(chunk_img, (target_w, target_h), interpolation=cv2.INTER_LANCZOS4)
                    h_chunk, w_chunk, _ = resized_chunk.shape
                    
                    y1_clip = min(y0 + h_chunk, canvas_h)
                    x1_clip = min(x0 + w_chunk, canvas_w)
                    w_clip = x1_clip - x0
                    h_clip = y1_clip - y0
                    
                    if w_clip > 0 and h_clip > 0:
                        current_roi = canvas[y0:y1_clip, x0:x1_clip]
                        incoming_chunk_roi = resized_chunk[0:h_clip, 0:w_clip]
                        
                        canvas[y0:y1_clip, x0:x1_clip] = np.minimum(current_roi, incoming_chunk_roi)
            
            stitched_output_path = os.path.join(page_folder, f"stitched_full_page_{page_num}.png")
            cv2.imwrite(stitched_output_path, canvas)
            print(f"    [➔] Generated Blended Page: stitched_full_page_{page_num}.png")
            print("-" * 85)

    print("="*85)
    print(f"[✓] Execution finished! Layer erasing bug resolved completely via Pixel-Minimum blending.")

if __name__ == "__main__":
    PDF_FILE_PATH = r"كشف حساب الراجحي.pdf"
    try_pdfimages_perfect_blend_stitching(PDF_FILE_PATH)